# Profiling — ANP Automotivos

Este notebook realiza o profiling das partições brutas da série histórica de preços de combustíveis automotivos da ANP.

O objetivo é compreender a estrutura e a qualidade da fonte antes da definição das regras de transformação da camada Silver.

São analisados:

- quantidade de linhas e colunas;
- intervalo das datas de coleta;
- produtos e unidades de medida;
- valores nulos;
- linhas completamente vazias;
- duplicatas exatas;
- colisões na combinação CNPJ da Revenda + Produto + Data da Coleta.

In [ ]:
from pathlib import Path

from insightfuel_data_platform.ingestion.anp import (
    carregar_particao,
    extrair_metadados_particao,
)
from insightfuel_data_platform.profiling.anp import (
    analisar_particao,
)

## 1. Descoberta das partições Bronze

Os arquivos CSV são armazenados na camada Bronze mantendo a organização por ano e semestre da fonte original.

Nesta etapa são descobertas todas as partições disponíveis para profiling.

In [2]:
pasta_bronze = Path("../data/bronze/anp/automotivos")

arquivos = sorted(
    pasta_bronze.rglob("*.csv")
)

print(f"Arquivos encontrados: {len(arquivos)}")

for arquivo in arquivos:
    print(arquivo)

Arquivos encontrados: 6
../data/bronze/anp/automotivos/ano=2023/semestre=1/AUTOMOTIVOS_2023.01.csv
../data/bronze/anp/automotivos/ano=2023/semestre=2/AUTOMOTIVOS_2023.02.csv
../data/bronze/anp/automotivos/ano=2024/semestre=1/AUTOMOTIVOS_2024.01.csv
../data/bronze/anp/automotivos/ano=2024/semestre=2/AUTOMOTIVOS_2024.02.csv
../data/bronze/anp/automotivos/ano=2025/semestre=1/AUTOMOTIVOS_2025.01.csv
../data/bronze/anp/automotivos/ano=2025/semestre=2/AUTOMOTIVOS_2025.02.csv


## 2. Profiling das partições

Cada partição é carregada individualmente e analisada pelas funções de profiling do projeto.

Os metadados de ano e semestre são extraídos da estrutura de diretórios da camada Bronze e adicionados ao resultado da análise.

In [3]:
resultados = []

for arquivo in arquivos:
    metadados = extrair_metadados_particao(arquivo)

    df = carregar_particao(arquivo)
    resultado = analisar_particao(df)

    resultado_completo = metadados | resultado
    resultados.append(resultado_completo)

## 3. Resumo do profiling

Os resultados abaixo consolidam as principais características de cada
partição analisada.

In [5]:
import polars as pl

df_resultados = pl.DataFrame(resultados)

df_resultados.select(
    "ano",
    "semestre",
    "total_linhas",
    "total_colunas",
    "data_minima",
    "data_maxima",
    "colisoes_chave_candidata",
    "duplicatas_exatas",
    "linhas_vazias",
)

ano,semestre,total_linhas,total_colunas,data_minima,data_maxima,colisoes_chave_candidata,duplicatas_exatas,linhas_vazias
i64,i64,i64,i64,date,date,i64,i64,i64
2023,1,431576,16,2023-01-02,2023-06-30,0,0,0
2023,2,472424,16,2023-07-03,2023-12-29,0,0,0
2024,1,477154,16,2024-01-01,2024-06-28,14,10,0
2024,2,421382,16,2024-07-01,2024-12-31,0,0,0
2025,1,429523,16,2025-01-01,2025-06-30,0,9113,9114
2025,2,384208,16,2025-07-01,2025-12-31,0,0,0


## 4. Valores nulos por partição

A análise de valores nulos auxilia na identificação de campos pouco preenchidos ou indisponíveis na fonte.

Em especial, `Valor de Compra` apresentou ausência total no período analisado e, por isso, não foi incorporado ao schema da camada Silver.

In [6]:
for resultado in resultados:
    print(
        f"{resultado['ano']}/{resultado['semestre']}:",
        resultado["nulos"],
    )

2023/1: {'Numero Rua': 107, 'Complemento': 333000, 'Bairro': 828, 'Valor de Compra': 431576}
2023/2: {'Numero Rua': 140, 'Complemento': 362579, 'Bairro': 854, 'Valor de Compra': 472424}
2024/1: {'Numero Rua': 65, 'Complemento': 366324, 'Bairro': 868, 'Valor de Compra': 477154}
2024/2: {'Numero Rua': 100, 'Complemento': 326684, 'Bairro': 573, 'Valor de Compra': 421382}
2025/1: {'Regiao - Sigla': 9114, 'Estado - Sigla': 9114, 'Municipio': 9114, 'Revenda': 9114, 'CNPJ da Revenda': 9114, 'Nome da Rua': 9114, 'Numero Rua': 9172, 'Complemento': 332908, 'Bairro': 9976, 'Cep': 9114, 'Produto': 9114, 'Data da Coleta': 9114, 'Valor de Venda': 9114, 'Valor de Compra': 429523, 'Unidade de Medida': 9114, 'Bandeira': 9114}
2025/2: {'Numero Rua': 24, 'Complemento': 297501, 'Bairro': 662, 'Valor de Compra': 384208}


## 5. Conclusões do profiling

O profiling das seis partições de 2023 a 2025 fundamentou as regras adotadas na camada Silver.

Principais conclusões:

- `Valor de Compra` está integralmente ausente no período analisado;
- foram identificadas linhas completamente vazias em 2025/1;
- foram identificadas duplicatas exatas em 2024/1;
- a combinação `CNPJ da Revenda + Produto + Data da Coleta` não constitui uma chave única confiável, pois existem observações distintas com a mesma combinação;
- a deduplicação deve considerar somente registros integralmente idênticos;
- CNPJ e CEP devem ser tratados como identificadores textuais;
- `Data da Coleta` deve ser convertida para tipo de data;
- `Valor de Venda` apresentou no máximo duas casas decimais;
- as unidades `R$ / m³` e `R$ / m3` devem ser padronizadas;
- município e UF serão utilizados para integração com o código oficial de município do IBGE.

As regras detalhadas e os resultados completos do profiling estão documentados em `docs/data-profiling/anp-automotivos.md`.